In [1]:
import pandas as pd
import polars as pl
from pathlib import Path
import util

pd.set_option('display.float_format', '{:,.1f}'.format)

## Regional Emissions
Only includes light, medium, and heavy vehicles (bus vehicles are excluded). TNC emissions can be included, depending on model settings reported below.

In [2]:
print(f"Include TNC & Taxi Emissions: {util.input_config['include_tnc_emissions']}")

Include TNC & Taxi Emissions: False


In [3]:
emissions_summary = pd.read_csv(util.output_path / 'emissions/emissions_summary.csv')

network = util.process_network_summary()

In [4]:
df_emissions_summary = emissions_summary.copy()

cols_dict = {'pollutant_name': 'Pollutant', 
             'veh_type': 'Vehicle Type',
             'start_tons': 'Start', 
             'intrazonal_tons': 'Intrazonal', 
             'interzonal_tons': 'Interzonal',
             'total_daily_tons': 'Total Daily (Tons)'}
cols = ['Start', 'Intrazonal','Interzonal', 'Total Daily (Tons)']
df_emissions_summary.rename(columns = cols_dict, inplace=True)


In [5]:
df = df_emissions_summary[df_emissions_summary['Vehicle Type'].isin(['light','medium','heavy'])].copy()
df = df.groupby('Pollutant').sum()
df.rename(columns = cols_dict, inplace=True)
df = df.loc[['CO','NOx','PM25 Total','PM10 Total','CO2 Equivalent','VOCs']]

# FIXME line below is failing at 3.11. I dont see a need for it since there are no decimals in the output.
#df = df.applymap(lambda x: x if x > 100 else str(round(x,1)))
df[cols]

,Start,Intrazonal,Interzonal,Total Daily (Tons)
Pollutant,,,,
CO,123.7,2.0,236.9,362.6
NOx,8.2,0.2,36.7,45.1
PM25 Total,0.4,0.0,1.3,1.7
PM10 Total,0.4,0.1,4.9,5.4
CO2 Equivalent,"2,298.3",253.0,"36,949.6","39,501.0"
VOCs,7.3,0.0,4.7,12.0


## Results by Vehicle Type

### VMT

In [6]:
df_network = network.copy()
if util.input_config['include_tnc']:
    df_network['@lveh'] = df_network[['@hov2_inc1','@hov2_inc2', '@hov2_inc3', 
                                      '@hov3_inc1', '@hov3_inc2', '@hov3_inc3',
                                      '@sov_inc1', '@sov_inc2', '@sov_inc3', 
                                      '@tnc_inc1', '@tnc_inc2','@tnc_inc3']].sum(axis=1)
else:
    df_network['@lveh'] = df_network[['@hov2_inc1','@hov2_inc2', '@hov2_inc3', 
                                      '@hov3_inc1', '@hov3_inc2', '@hov3_inc3',
                                      '@sov_inc1', '@sov_inc2', '@sov_inc3']].sum(axis=1)

df_network['light'] = df_network['@lveh']*df_network['length']
df_network['medium'] = df_network['@mveh']*df_network['length']
df_network['heavy'] = df_network['@hveh']*df_network['length']

index_labels = ['light','medium','heavy']
df = pd.DataFrame(index=index_labels)
df['VMT'] = df_network[index_labels].sum()

df.index.name = 'Vehicle Type'
df

,VMT
Vehicle Type,
light,"77,164,942.5"
medium,"3,215,271.1"
heavy,"2,604,497.0"


### Emissions

In [7]:
# Calculate emissions and VMT by vehicle type and save results
# Note that Total VMT will not match regional totals because we are not included buses in the emissions summaries

df = df_emissions_summary.copy()
df = df.groupby(['Pollutant','Vehicle Type']).sum()
df.rename(columns = cols_dict, inplace=True)

df.loc[['CO','NOx','PM25 Total','PM10 Total','CO2 Equivalent','VOCs']][cols].copy()


Start  Intrazonal  Interzonal  \
Pollutant      Vehicle Type                                   
CO             heavy            0.0         0.0         6.8   
               light          115.6         2.0       220.3   
               medium           8.1         0.0         9.8   
               transit          0.0         0.0         1.5   
NOx            heavy            0.0         0.0        15.7   
               light            7.3         0.1        18.2   
               medium           0.9         0.0         2.7   
               transit          0.0         0.0         0.6   
PM25 Total     heavy            0.0         0.0         0.4   
               light            0.3         0.0         0.8   
               medium           0.0         0.0         0.1   
               transit          0.0         0.0         0.0   
PM10 Total     heavy            0.0         0.0         0.7   
               light            0.4         0.0         3.9   
               medium           0.0         0.0         0.3   
               transit          0.0         0.0         0.0   
CO2 Equivalent heavy            2.8         3.5     4,940.9   
               light        2,167.2       247.8    29,664.6   
               medium         128.4         1.7     2,344.2   
               transit          1.0         0.0       338.1   
VOCs           heavy            0.0         0.0         0.5   
               light            6.7         0.0         3.8   
               medium           0.5         0.0         0.3   
               transit          0.0         0.0         0.0   

                             Total Daily (Tons)  
Pollutant      Vehicle Type                      
CO             heavy                        6.9  
               light                      337.9  
               medium                      17.9  
               transit                      1.5  
NOx            heavy                       15.8  
               light                       25.6  
               medium                       3.6  
               transit                      0.6  
PM25 Total     heavy                        0.4  
               light                        1.2  
               medium                       0.1  
               transit                      0.0  
PM10 Total     heavy                        0.7  
               light                        4.3  
               medium                       0.3  
               transit                      0.0  
CO2 Equivalent heavy                    4,947.1  
               light                   32,079.6  
               medium                   2,474.3  
               transit                    339.1  
VOCs           heavy                        0.5  
               light                       10.6  
               medium                       0.9  
               transit                      0.0